In [1]:
import numpy as np
import h5py
from model_ranking import ensure_even_label_sampling

INFO: P [MainThread] 2026-01-28 13:08:56,282 plantseg - Logger configured at initialisation. PlantSeg logger name: plantseg


/g/kreshuk/talks/pytorch-3dunet/pytorch3dunet/unet3d/utils.py:17: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/g/kreshuk/talks/miniforge3/envs/model-rank-local2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path = "/g/kreshuk/talks/sampled_features/semantic_segmentation/mitochondria/1k_pixels_sampled/EPFL_to_EPFL/E_model_NA2_to_EPFL_features.h5"
layer_key = "decoders.2"
with h5py.File(path, "r") as f:
    features = f[f"{layer_key}_features"][:]
    labels = f[f"{layer_key}_labels"][:]
    predictions = f[f"{layer_key}_predictions"][:]

In [3]:
features_balanced, labels_balanced, predictions_balanced = ensure_even_label_sampling(
    features, labels, predictions, n_samples_per_class=None
)

Original dataset: 750000 samples
Balanced dataset: 644668 samples


In [8]:
predictions_balanced.shape

(644668,)

In [4]:
# invert the sigmoid on the predictions by applying the logit funciton
predictions_balanced = np.clip(predictions_balanced, 1e-7, 1 - 1e-7)
predictions_balanced_logits = np.log(predictions_balanced / (1 - predictions_balanced))

In [5]:
predictions_balanced_logits[:3]

array([-10.498209,  -9.016449,  -8.792429], dtype=float32)

In [6]:
from model_ranking import uniform_cross_entropy, MaNo_evaluate
import torch
import torch.nn as nn


In [7]:
# For binary classification, we need 2-class logits
# Stack negative and positive logits
logits_2class = np.stack([np.zeros_like(predictions_balanced_logits), predictions_balanced_logits], axis=1)
delta = uniform_cross_entropy(2, logits_2class)
print(delta)

tensor(3.6716, device='cuda:0')


In [8]:
logits_rel = np.stack([-predictions_balanced_logits/2, predictions_balanced_logits/2], axis=1)

In [11]:
delta = uniform_cross_entropy(2, logits_2class)
print(delta)

tensor(3.6716, device='cuda:0')


In [10]:
delta_rel = uniform_cross_entropy(2, logits_rel)
print(delta_rel)

tensor(3.6716, device='cuda:0')


In [12]:
MaNo_score = MaNo_evaluate(4, float(delta.item()), logits_rel)

In [13]:
MaNo_score

0.62174994

In [14]:
from model_ranking import scaling_method

In [15]:
logits_scaled_rel = scaling_method(torch.tensor(logits_rel), delta_rel)

In [17]:
logits_scaled_rel[:3]

tensor([[0.6776, 0.3224],
        [0.7019, 0.2981],
        [0.7061, 0.2939]])

In [18]:
logits_scaled = scaling_method(torch.tensor(logits_2class), delta)

In [19]:
logits_scaled[:3]

tensor([[0.0215, 0.9785],
        [0.0297, 0.9703],
        [0.0314, 0.9686]])

In [20]:
predictions_balanced[:3]

array([2.7585051e-05, 1.2138171e-04, 1.5185554e-04], dtype=float32)

In [21]:
preds_2class = np.stack([1-predictions_balanced, predictions_balanced], axis=1)
print(preds_2class[:3])

[[9.9997240e-01 2.7585051e-05]
 [9.9987864e-01 1.2138171e-04]
 [9.9984813e-01 1.5185554e-04]]
